# Milestone 1 - Baseline BM25 no BioASQ

Este notebook documenta a primeira milestone do projeto **MedIR-RAG**.

Objetivos:

- carregar o dataset BioASQ;
- construir queries, documentos e julgamentos de relevância (`qrels`);
- aplicar tokenização, normalização e remoção de stop-words;
- criar uma baseline lexical com BM25;
- avaliar a ordenação com `NDCG@10`, `MAP` e `MRR`.

A implementação será executada por etapas para tornar visível o percurso dos dados.

## 1. Configuração

Começamos por importar as bibliotecas. Nesta execução experimental usamos o dataset configurado na célula seguinte, que está dentro desta pasta.

Este dataset é adequado para testar o pipeline rapidamente. Para os resultados finais, devemos indicar no relatório qual dataset foi usado e manter a mesma coleção em todas as comparações.

In [1]:
from pathlib import Path
import json
import string
from collections import Counter

import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from rank_bm25 import BM25Okapi
from scipy.sparse import csr_matrix
import pytrec_eval

DATASET_PATH = Path("BioASQ14-2026-Training-14b.json")

print(f"Dataset: {DATASET_PATH.resolve()}")
print(f"Existe: {DATASET_PATH.exists()}")

Dataset: C:\Users\diogu\Documents\1uni\MEI\RIBT\RIBT_Projeto_MedIR_RAG\Milestone_1\BioASQ14-2026-Training-14b.json
Existe: True


## 2. Carregamento e inspeção do dataset

O BioASQ contém perguntas, documentos relevantes e snippets associados a artigos do PubMed.

In [2]:
with DATASET_PATH.open("r", encoding="utf-8") as file:
    data = json.load(file)

questions = data["questions"]

print(f"Número de registos: {len(questions)}")
print("Campos da primeira pergunta:")
print(questions[0].keys())
print("\nPrimeira pergunta:")
print(questions[0]["body"])
print(f"\nNúmero de snippets: {len(questions[0].get('snippets', []))}")

Número de registos: 5729
Campos da primeira pergunta:
dict_keys(['body', 'documents', 'ideal_answer', 'concepts', 'type', 'id', 'snippets'])

Primeira pergunta:
Is Hirschsprung disease a mendelian or a multifactorial disorder?

Número de snippets: 16


## 3. Pré-processamento

Nesta baseline vamos usar minúsculas, remoção de pontuação, tokenização e stop-words em inglês. O objetivo é fazer com que palavras como `Cancer` e `cancer` sejam tratadas da mesma forma.

In [3]:
try:
    stopwords.words("english")
except LookupError:
    nltk.download("stopwords")

try:
    word_tokenize("test")
except LookupError:
    nltk.download("punkt")
    nltk.download("punkt_tab")

STOP_WORDS = set(stopwords.words("english"))


def preprocess_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    tokens = word_tokenize(text)
    return [token for token in tokens if token not in STOP_WORDS and token.strip()]

example_text = questions[0]["body"]
print("Antes:")
print(example_text)
print("\nDepois:")
print(preprocess_text(example_text))

Antes:
Is Hirschsprung disease a mendelian or a multifactorial disorder?

Depois:
['hirschsprung', 'disease', 'mendelian', 'multifactorial', 'disorder']


## 4. Construção das queries, corpus e qrels

- `queries`: pergunta usada para pesquisar;
- `corpus`: documentos que o BM25 vai indexar, construídos a partir dos snippets;
- `qrels`: documentos considerados relevantes para cada pergunta.

In [4]:
queries = {}
corpus = {}
qrels = {}

for question in questions:
    question_id = question.get("id")
    question_text = question.get("body")

    if not question_id or not question_text:
        continue

    queries[question_id] = question_text
    qrels[question_id] = {
        document_id: 1
        for document_id in question.get("documents", [])
    }

    for snippet in question.get("snippets", []):
        document_id = snippet.get("document")
        snippet_text = snippet.get("text", "")

        if not document_id or not snippet_text:
            continue

        corpus[document_id] = corpus.get(document_id, "") + " " + snippet_text

print(f"Queries: {len(queries)}")
print(f"Documentos antes da tokenização: {len(corpus)}")
print(f"Julgamentos de relevância: {len(qrels)}")

Queries: 5729
Documentos antes da tokenização: 48949
Julgamentos de relevância: 5729


## 5. Tokenização do corpus e das queries

O BM25 recebe listas de tokens, não texto bruto. Por isso aplicamos a mesma função de pré-processamento às perguntas e aos snippets.

In [5]:
tokenized_queries = {
    question_id: preprocess_text(question_text)
    for question_id, question_text in queries.items()
}

tokenized_corpus = {
    document_id: preprocess_text(document_text)
    for document_id, document_text in corpus.items()
}

doc_ids = list(tokenized_corpus)
corpus_tokens = [tokenized_corpus[document_id] for document_id in doc_ids]

print(f"Documentos tokenizados: {len(corpus_tokens)}")
print(f"Tokens no primeiro documento: {len(corpus_tokens[0])}")
print(f"Primeiros tokens: {corpus_tokens[0][:20]}")

Documentos tokenizados: 48949
Tokens no primeiro documento: 60
Primeiros tokens: ['hirschsprung', 'disease', 'hscr', 'multifactorial', 'nonmendelian', 'disorder', 'rare', 'highpenetrance', 'coding', 'sequence', 'mutations', 'receptor', 'tyrosine', 'kinase', 'ret', 'contribute', 'risk', 'combination', 'mutations', 'genes']


## 6. Indexação BM25

O BM25 calcula uma pontuação de relevância entre uma query e cada documento. Quanto maior a pontuação, mais compatível é o documento com a pergunta.

In [6]:
BM25_K1 = 0.60
BM25_B = 0.75
bm25 = BM25Okapi(corpus_tokens, k1=BM25_K1, b=BM25_B)

# Build the BM25 weights once so query scoring uses sparse matrix operations
# instead of iterating over every document in Python for every query.
term_to_index = {
    term: index
    for index, term in enumerate(bm25.idf)
}
rows = []
columns = []
weights = []

for document_index, frequencies in enumerate(bm25.doc_freqs):
    document_length_factor = bm25.k1 * (
        1 - bm25.b + bm25.b * bm25.doc_len[document_index] / bm25.avgdl
    )
    for term, frequency in frequencies.items():
        rows.append(document_index)
        columns.append(term_to_index[term])
        weights.append(
            bm25.idf[term]
            * frequency
            * (bm25.k1 + 1)
            / (frequency + document_length_factor)
        )

bm25_weight_matrix = csr_matrix(
    (weights, (rows, columns)),
    shape=(len(doc_ids), len(term_to_index)),
    dtype=np.float64,
)


def get_bm25_scores(tokenized_query):
    query_counts = Counter(
        term for term in tokenized_query if term in term_to_index
    )
    if not query_counts:
        return np.zeros(len(doc_ids), dtype=np.float64)

    term_indexes = [term_to_index[term] for term in query_counts]
    term_counts = np.fromiter(query_counts.values(), dtype=np.float64)
    return np.asarray(
        bm25_weight_matrix[:, term_indexes].dot(term_counts)
    ).ravel()

print("Índice BM25 criado.")

Índice BM25 criado.


## 7. Primeira pesquisa

Antes de calcular métricas, vamos observar manualmente os documentos mais bem classificados para uma pergunta.

In [7]:
example_query_id = next(iter(tokenized_queries))
example_query = tokenized_queries[example_query_id]
example_scores = get_bm25_scores(example_query)

ranked_indexes = example_scores.argsort()[::-1][:5]

print(f"Query: {queries[example_query_id]}")
print("\nTop 5 documentos:")

for rank, index in enumerate(ranked_indexes, start=1):
    document_id = doc_ids[index]
    print(f"\n{rank}. score={example_scores[index]:.4f}")
    print(f"ID: {document_id}")
    print(corpus[document_id][:400], "...")

Query: Is Hirschsprung disease a mendelian or a multifactorial disorder?

Top 5 documentos:

1. score=22.6323
ID: http://www.ncbi.nlm.nih.gov/pubmed/15829955
 Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the receptor tyrosine kinase RET contribute to risk in combination with mutations at other genes  Hirschsprung disease (HSCR) is a multifactorial, non-mendelian disorder in which rare high-penetrance coding sequence mutations in the receptor tyrosine kinase RET contri ...

2. score=20.0193
ID: http://www.ncbi.nlm.nih.gov/pubmed/6650562
  The inheritance of Hirschsprung disease is generally consistent with sex-modified multifactorial inheritance with a lower threshold of expression in males. The inheritance of Hirschsprung disease is generally consistent with sex-modified multifactorial inheritance with a lower threshold of expression in males ...

3. score=18.8678
ID: http://www.ncbi.nlm.nih.gov/pubme

## 8. Avaliação quantitativa

A avaliação usa apenas queries que têm pelo menos um documento relevante presente no corpus indexado.

Além das métricas pedidas inicialmente, calculamos as métricas indicadas no enunciado completo: `NDCG@5`, `NDCG@10`, `MRR@10`, `MAP` e `Recall@100`.

In [8]:
indexed_qrels = {
    question_id: {
        document_id: relevance
        for document_id, relevance in judgments.items()
        if document_id in tokenized_corpus
    }
    for question_id, judgments in qrels.items()
}
indexed_qrels = {
    question_id: judgments
    for question_id, judgments in indexed_qrels.items()
    if judgments
}

metrics = {
    "ndcg_cut_5",
    "ndcg_cut_10",
    "map",
    "recip_rank",
    "recall_100",
}

evaluator = pytrec_eval.RelevanceEvaluator(indexed_qrels, metrics)
metric_sums = {metric: 0.0 for metric in metrics}
mrr_10_sum = 0.0
number_of_queries = 0
full_run = {}
top_10_indexes = {}

for question_id, tokenized_query in tokenized_queries.items():
    if question_id not in indexed_qrels:
        continue

    scores = get_bm25_scores(tokenized_query)
    full_run[question_id] = {
        document_id: float(score)
        for document_id, score in zip(doc_ids, scores)
    }

    ranked_indexes = np.argpartition(scores, -10)[-10:]
    ranked_indexes = ranked_indexes[np.argsort(scores[ranked_indexes])[::-1]]
    top_10_indexes[question_id] = ranked_indexes
    number_of_queries += 1

if number_of_queries == 0:
    raise ValueError("Nenhuma query pode ser avaliada.")

evaluation = evaluator.evaluate(full_run)
for result in evaluation.values():
    for metric in metric_sums:
        metric_sums[metric] += result[metric]

for question_id, ranked_indexes in top_10_indexes.items():
    mrr_10_sum += next(
        (
            1.0 / rank
            for rank, index in enumerate(ranked_indexes, start=1)
            if doc_ids[index] in indexed_qrels[question_id]
        ),
        0.0,
    )

results = {
    "NDCG@5": metric_sums["ndcg_cut_5"] / number_of_queries,
    "NDCG@10": metric_sums["ndcg_cut_10"] / number_of_queries,
    "MRR@10": mrr_10_sum / number_of_queries,
    "MAP": metric_sums["map"] / number_of_queries,
    "Recall@100": metric_sums["recall_100"] / number_of_queries,
}

print(f"Queries avaliadas: {number_of_queries}")
for metric_name, value in results.items():
    print(f"{metric_name:10s}: {value:.4f}")

Queries avaliadas: 5729
NDCG@5    : 0.7160
NDCG@10   : 0.7016
MRR@10    : 0.8205
MAP       : 0.6304
Recall@100: 0.8396


## 9. Interpretação

Os valores finais devem ser interpretados em conjunto:

- `NDCG@k`: avalia a qualidade da ordenação até à posição `k`;
- `MAP`: mede a precisão média ao longo da lista recuperada;
- `MRR@10`: verifica quão cedo aparece o primeiro documento relevante;
- `Recall@100`: mede quanto dos documentos relevantes foi recuperado nos primeiros 100 resultados.

Estes resultados constituem a baseline lexical que será comparada com os modelos densos na Milestone 2.

## 10. Exportação dos resultados

Para além do output visível no notebook, guardamos as métricas num ficheiro JSON. Assim podemos reutilizá-las no relatório e comparar esta baseline com os modelos da Milestone 2.

In [9]:
results_path = Path(f"bm25_results_{DATASET_PATH.stem}.json")

with results_path.open("w", encoding="utf-8") as file:
    json.dump(
        {
            "dataset": str(DATASET_PATH),
            "queries_evaluated": number_of_queries,
            "metrics": results,
        },
        file,
        indent=2,
    )

print(f"Resultados guardados em: {results_path.resolve()}")

Resultados guardados em: C:\Users\diogu\Documents\1uni\MEI\RIBT\RIBT_Projeto_MedIR_RAG\Milestone_1\bm25_results_BioASQ14-2026-Training-14b.json
